# GPU run on Kaggle — MagnaTagATune + GTZAN

Same experiments as `colab_run.ipynb`, trimmed to **only what carries marks**
so it finishes in about 2 hours instead of most of a day.

**Before you start — three settings in the right-hand panel:**

1. **Accelerator → GPU T4 x2** (or P100)
2. **Internet → On** &nbsp;← *without this, nothing downloads and every cell fails*
3. **Persistence → Files only** (optional, lets you resume)

Kaggle gives 30 GPU-hours a week and 9 hours per session, so this fits easily.

## What runs

| | Corpus | Why |
|---|---|---|
| Baselines B1/B2/B4 | MagnaTagATune | required comparison |
| Task 1 — BERT tagging | MagnaTagATune | spec's corpus for Task 1 |
| Task 3 — fusion, 4-arm ablation | MagnaTagATune | spec's corpus for Task 3 |
| Task 4 — contrastive retrieval | MagnaTagATune | textless clips dropped |
| Task 2 — genre classification | GTZAN | spec's corpus for Task 2 |

Optional extras (GAT, chord-graph comparison, DEAM, MusicCaps) are at the end,
switched off by default. Turn them on only if you have time to spare.

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. Right-hand panel > Accelerator > GPU T4 x2, then re-run.")

## 1. Get the code

Kaggle's writable directory is `/kaggle/working`. Everything below lives there.

In [ ]:
REPO = "sharminahmednova/Supervised-Neural-Network-Project-GNN-Based-BERT-for-Understanding-Context-from-Music"
TOKEN = ""    # only needed if the repo is PRIVATE

REPO_URL = (f"https://{TOKEN}@github.com/{REPO}.git" if TOKEN
            else f"https://github.com/{REPO}.git")
WORKDIR = "/kaggle/working/project"

import os, subprocess
from pathlib import Path

if (Path(WORKDIR) / ".git").exists():
    subprocess.run(["git", "-C", WORKDIR, "pull", "-q"], check=False)
    print("updated existing clone")
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, WORKDIR], check=True)
    print("cloned")

os.chdir(WORKDIR)
print("cwd:", Path.cwd())
subprocess.run(["ls"])

If the clone fails with a DNS or connection error, **Internet is off**. Turn it
on in the right-hand panel (it requires a phone-verified Kaggle account) and
re-run this cell.

## 2. Dependencies

Kaggle already ships a CUDA torch; only the rest is installed.

In [ ]:
!pip install -q torch-geometric==2.8.0.post1 transformers==5.17.0 librosa==0.11.0 soundfile==0.14.0 pyyaml==6.0.3
!python tools/check_env.py

## 3. Fetch the corpora

MagnaTagATune is ~2.8 GB and GTZAN ~1.1 GB. Kaggle's `/kaggle/working` holds
20 GB, which is enough for both plus the derived graphs.

In [ ]:
!python tools/fetch_data.py magnatagatune --extract
!python tools/fetch_data.py gtzan --extract
!du -sh data/raw/*

## 4. MagnaTagATune — baselines, Task 1, Task 3

`SUBSAMPLE` is the main time dial. 4,000 clips takes roughly 90 minutes end to
end and is plenty for the report; raise it to 8,000 if you have hours to spare.

In [ ]:
SUBSAMPLE = 4000
EPOCHS    = 20
BATCH     = 64

COMMON = ["--config", "configs/mtat.yaml", "--set", "device=cuda",
          "--set", f"train.epochs={EPOCHS}", "--set", f"train.batch_size={BATCH}",
          "--set", "train.patience=6", "--set", "text.unfreeze_last_n=4"]

import subprocess, time

def run(cmd, label):
    print(f"\n{'='*70}\n{label}\n{'='*70}", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd)
    print(f"[{label}] {'OK' if r.returncode == 0 else 'FAILED'} "
          f"in {(time.time()-t0)/60:.1f} min", flush=True)
    return r.returncode == 0

In [ ]:
run(["python", "src/preprocess.py", "--config", "configs/mtat.yaml",
     "--set", f"dataset.subsample={SUBSAMPLE}", "--export-samples", "20"],
    "preprocess MagnaTagATune")

In [ ]:
run(["python", "src/train.py", "baselines", *COMMON], "baselines (B1, B2, B4)")

In [ ]:
run(["python", "src/train.py", "task1", *COMMON], "Task 1 - BERT tagging")

In [ ]:
# The 4-arm ablation: bert_only / gnn_only / concat / cross_attention.
# This is the longest cell -- roughly 45 min at SUBSAMPLE=4000.
run(["python", "src/train.py", "task3", "--ablation", *COMMON],
    "Task 3 - fusion ablation")

### Task 4 — retrieval, on its own preprocessing

~35% of MagnaTagATune clips have no text-side tag after the disjoint tag split
and share one placeholder string. Identical text ties exactly in retrieval, so
those clips are dropped here. Writes to `results_mtat_r4`.

In [ ]:
R4 = ["--set", "dataset.drop_textless=true",
      "--set", "paths.processed=data/processed_mtat_r4",
      "--set", "paths.splits=data/splits_mtat_r4"]

if run(["python", "src/preprocess.py", "--config", "configs/mtat.yaml",
        "--set", f"dataset.subsample={SUBSAMPLE}", *R4, "--export-samples", "0"],
       "preprocess MTAT for retrieval"):
    run(["python", "src/train.py", "task4", *COMMON, *R4,
         "--set", "paths.results=results_mtat_r4"],
        "Task 4 - contrastive retrieval")

## 5. GTZAN — Task 2 genre classification

Fast: 1,000 clips, no BERT in the loop.

In [ ]:
GTZAN = ["--config", "configs/gtzan.yaml", "--set", "device=cuda",
         "--set", f"train.epochs={EPOCHS}", "--set", f"train.batch_size={BATCH}"]

run(["python", "src/preprocess.py", "--config", "configs/gtzan.yaml",
     "--export-samples", "20"], "preprocess GTZAN")
run(["python", "src/train.py", "baselines", *GTZAN], "GTZAN baselines")
run(["python", "src/train.py", "task2", *GTZAN], "Task 2 - GNN genre classification")

## 6. Tables and plots

In [ ]:
!python src/evaluate.py --set paths.results=results_mtat --set paths.plots=results_mtat/plots
!python src/evaluate.py --set paths.results=results_gtzan --set paths.plots=results_gtzan/plots
!python tools/make_report_tables.py --results results_mtat
!cat results_mtat/comparison_table.txt

## 7. Package the results

Kaggle has no `files.download()`. Instead, write the zip to `/kaggle/working`
and grab it from the **Output** tab on the right (or *Save Version → Output*).

In [ ]:
import shutil, subprocess
from pathlib import Path

OUT = Path("/kaggle/working/results_gpu")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

for name in ["results_mtat", "results_mtat_r4", "results_gtzan"]:
    src = Path(name)
    if src.exists():
        shutil.copytree(src, OUT / name, dirs_exist_ok=True)

# The 20 sample graphs and the corpus metadata are submission items.
for proc in ["data/processed_mtat", "data/processed_gtzan"]:
    p = Path(proc)
    if not p.exists():
        continue
    dst = OUT / proc
    dst.mkdir(parents=True, exist_ok=True)
    if (p / "samples").exists():
        shutil.copytree(p / "samples", dst / "samples", dirs_exist_ok=True)
    for meta in ["meta.json", "label_space.json", "index.json"]:
        if (p / meta).exists():
            shutil.copy2(p / meta, dst / meta)

shutil.make_archive("/kaggle/working/results_gpu", "zip", OUT)
shutil.rmtree(OUT)
size = Path("/kaggle/working/results_gpu.zip").stat().st_size / 1e6
print(f"results_gpu.zip  ({size:.1f} MB)")
print("Download it from the Output panel on the right.")

## 8. Optional extras

Everything above is what the marks depend on. These add breadth if you have
session time left — flip a flag to `True` and re-run the cell.

In [ ]:
RUN_GAT          = False   # GAT instead of GraphSAGE on Task 2
RUN_GRAPH_KINDS  = False   # segment vs chord vs hybrid graphs
RUN_DEAM         = False   # valence/arousal auxiliary loss (~25 min, +1.7 GB)
RUN_MUSICCAPS    = False   # Task 4 on real captions (slow, YouTube often blocks)

if RUN_GAT:
    run(["python", "src/train.py", "task2", *GTZAN, "--set", "gnn.conv=gat",
         "--set", "paths.results=results_gtzan_gat"], "Task 2 with GAT")

if RUN_GRAPH_KINDS:
    for kind in ["segment", "chord", "hybrid"]:
        ok = run(["python", "src/preprocess.py", "--config", "configs/gtzan.yaml",
                  "--set", f"graph.kind={kind}",
                  "--set", f"paths.processed=data/processed_gtzan_{kind}",
                  "--set", f"paths.splits=data/splits_gtzan_{kind}",
                  "--export-samples", "0"], f"preprocess GTZAN ({kind})")
        if ok:
            run(["python", "src/train.py", "task2", *GTZAN,
                 "--set", f"graph.kind={kind}",
                 "--set", f"paths.processed=data/processed_gtzan_{kind}",
                 "--set", f"paths.splits=data/splits_gtzan_{kind}",
                 "--set", f"paths.results=results_gtzan_{kind}"],
                f"Task 2 ({kind} graph)")

if RUN_DEAM:
    if run(["python", "tools/fetch_data.py", "deam", "--extract"], "fetch DEAM"):
        run(["python", "src/preprocess.py", "--config", "configs/deam.yaml",
             "--export-samples", "20"], "preprocess DEAM")
        run(["python", "src/train.py", "task3", "--config", "configs/deam.yaml",
             "--mode", "cross_attention", "--set", "device=cuda",
             "--set", f"train.epochs={EPOCHS}", "--set", f"train.batch_size={BATCH}"],
            "Task 3 with DEAM emotion target")

if RUN_MUSICCAPS:
    !pip install -q yt-dlp
    run(["python", "tools/fetch_data.py", "musiccaps", "--clips", "800"],
        "fetch MusicCaps")

## 9. The listening study

Build it here so the clips come from the same run as your numbers, then
download `rating_form.html` from the Output panel and send it to 5 people.

In [ ]:
import os
from pathlib import Path

RES, PROC = next(((r, p) for r, p in [
    ("results_musiccaps", "data/processed_musiccaps"),
    ("results_mtat_r4",   "data/processed_mtat_r4"),
    ("results_mtat",      "data/processed_mtat"),
] if Path(r, "retrieval_examples").exists()), (None, None))

assert RES, "no Task 4 retrieval output -- run the Task 4 cell first"
print("building the study from", RES)

!python tools/make_human_eval.py --results $RES --processed $PROC --queries 10 --seconds 12
!cp $RES/human_eval/rating_form.html /kaggle/working/rating_form.html
print("\nrating_form.html is in the Output panel -- send that one file to 5 listeners.")

## Back on your machine

Download **`results_gpu.zip`** and **`rating_form.html`** from the Output panel,
put the zip in your project folder, then:

```bash
unzip -o results_gpu.zip -d .
python src/evaluate.py --set paths.results=results_mtat --set paths.plots=results_mtat/plots
python tools/make_report_tables.py --results results_mtat
```